# RainGuard — Module 5: Regression Model
**Indian Rainfall Forecasting and Heavy Rainfall Risk Analysis System**

**Goal:** Forecast next-calendar-day district rainfall (`target_next_mm`) using only information available at the end of the current day. This notebook uses the engineered table and feature groups saved by Module 4.

**Models:** persistence baseline, Linear Regression, Random Forest Regressor, Gradient Boosting Regressor.

**Evaluation:** MAE, MSE, RMSE and R². A chronological split is used so future dates are never used to train a model evaluated on earlier dates.

## Step 1 — Imports, paths and reproducibility

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

cwd = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(cwd, "..")) if os.path.basename(cwd) == "notebooks" else cwd
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "rainfall_features.csv")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")
FIG_DIR = os.path.join(PROJECT_ROOT, "docs", "figures")
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

assert os.path.exists(DATA_PATH), f"Feature dataset not found: {DATA_PATH}. Run Module 4 first."
assert os.path.exists(os.path.join(MODEL_DIR, "feature_config.json")), "feature_config.json not found. Run Module 4 first."

with open(os.path.join(MODEL_DIR, "feature_config.json"), encoding="utf-8") as f:
    config = json.load(f)

print("Imports and paths ready")
print("Project root:", PROJECT_ROOT)
print("Feature data:", DATA_PATH)

## Step 2 — Load and validate the Module 4 modelling table
We use the exact target and feature groups created in Module 4. We do not rebuild or modify the target here.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["date", "target_date"])
TARGET = config["targets"]["regression"]
FEATURE_GROUPS = config["feature_groups"]

print("Shape:", df.shape)
print("Districts:", df["district"].nunique())
print("Forecast dates:", df["target_date"].min().date(), "to", df["target_date"].max().date())
print("Target:", TARGET)
print("Missing target values:", df[TARGET].isna().sum())
print("Negative target values:", (df[TARGET] < 0).sum())
display(df.head())

assert TARGET in df.columns
assert df[TARGET].notna().all(), "Target has missing values."
assert (df[TARGET] >= 0).all(), "Rainfall target cannot be negative."
assert (df["target_date"] - df["date"]).dt.days.eq(1).all(), "Target date is not exactly one day after feature date."
assert not df.duplicated(["state", "district", "date"]).any(), "Duplicate district-date rows found."
print("Validation passed.")

## Step 3 — Define the candidate feature sets
Module 4 documented five groups. We compare a core history model with progressively expanded feature sets. The numeric location codes are included only in tree models as an experiment; their integer ordering has no natural meaning for linear regression, so they are excluded from the primary linear feature set.

In [ ]:
history = FEATURE_GROUPS["history"]
season = FEATURE_GROUPS["season"]
location_numeric = FEATURE_GROUPS["location_numeric"]
location_codes = FEATURE_GROUPS["location_codes"]
climatology = FEATURE_GROUPS["climatology"]

FEATURE_SETS = {
    "History only": history,
    "History + season + district mean": history + season + location_numeric,
    "History + season + district mean + climatology": history + season + location_numeric + climatology,
    "Expanded (including location codes)": history + season + location_numeric + climatology + location_codes,
}

for name, cols in FEATURE_SETS.items():
    missing = [c for c in cols if c not in df.columns]
    assert not missing, f"{name}: missing columns {missing}"
    print(f"{name}: {len(cols)} features -> {cols}")

## Step 4 — Chronological train-test split
We split by **forecast date**, not by randomly shuffled rows. The earliest 80% of unique target dates are used for training and the latest 20% for testing. All districts on a given date stay on the same side of the split.

In [ ]:
unique_dates = np.array(sorted(df["target_date"].dropna().unique()))
split_pos = int(len(unique_dates) * 0.80)
split_pos = min(max(split_pos, 1), len(unique_dates) - 1)
split_date = pd.Timestamp(unique_dates[split_pos])

train_df = df[df["target_date"] < split_date].copy()
test_df = df[df["target_date"] >= split_date].copy()

print("Unique forecast dates:", len(unique_dates))
print("Training forecast dates:", train_df["target_date"].min().date(), "to", train_df["target_date"].max().date())
print("Testing forecast dates:", test_df["target_date"].min().date(), "to", test_df["target_date"].max().date())
print("Train rows:", len(train_df), "| Test rows:", len(test_df))
assert train_df["target_date"].max() < test_df["target_date"].min()
assert len(train_df) > 0 and len(test_df) > 0

# Save the split boundary for reproducibility and later modules
config["regression_split_date"] = str(split_date.date())
with open(os.path.join(MODEL_DIR, "feature_config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

## Step 5 — Establish a simple persistence baseline
The persistence baseline predicts tomorrow will have the same rainfall as today (`rain_today`). This is a useful reference: machine-learning models should be compared against a simple operational benchmark.

In [ ]:
y_train = train_df[TARGET].astype(float)
y_test = test_df[TARGET].astype(float)

def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred),
    }

baseline_pred = test_df["rain_today"].astype(float).clip(lower=0)
baseline_metrics = regression_metrics(y_test, baseline_pred)
print("Persistence baseline:")
display(pd.DataFrame([baseline_metrics], index=["Predict tomorrow = today"]).round(4))

## Step 6 — Train and compare the three regression algorithms
All preprocessing is fitted on training data only. The median imputer and scaler are placed inside the Linear Regression pipeline. Tree models use median imputation and do not need feature scaling. We compare each model using the documented feature sets.

In [ ]:
models = {
    "Linear Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=250, max_features=0.8, min_samples_leaf=2,
            random_state=RANDOM_STATE, n_jobs=-1
        )),
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(
            n_estimators=150, learning_rate=0.05, max_depth=2,
            loss="huber", random_state=RANDOM_STATE
        )),
    ]),
}

results = []
trained = {}
predictions = {}

for feature_set_name, features in FEATURE_SETS.items():
    X_train = train_df[features]
    X_test = test_df[features]
    for model_name, estimator in models.items():
        estimator.fit(X_train, y_train)
        pred = np.maximum(estimator.predict(X_test), 0)
        metrics = regression_metrics(y_test, pred)
        results.append({"Feature set": feature_set_name, "Model": model_name, **metrics})
        trained[(feature_set_name, model_name)] = estimator
        predictions[(feature_set_name, model_name)] = pred
        print(f"Finished: {feature_set_name} | {model_name}")

results_df = pd.DataFrame(results).sort_values(["RMSE", "MAE"]).reset_index(drop=True)
display(results_df.round(4))

## Step 7 — Select the best model using validation metrics
The primary selection criterion is the lowest test RMSE, with MAE and R² considered as supporting metrics. Because the test period is short, treat the ranking as an initial estimate rather than proof of general performance.

In [ ]:
best_row = results_df.iloc[0]
best_key = (best_row["Feature set"], best_row["Model"])
best_model = trained[best_key]
best_features = FEATURE_SETS[best_key[0]]
best_pred = predictions[best_key]

print("Best feature set:", best_key[0])
print("Best algorithm:", best_key[1])
print("Features:", best_features)
print("Metrics:")
display(pd.DataFrame([best_row[ ["MAE", "MSE", "RMSE", "R2"] ].to_dict()], index=[best_key[1]]).round(4))

## Step 8 — Visualize model comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_df = results_df.copy()
plot_df["Label"] = plot_df["Model"] + "\n" + plot_df["Feature set"].str.replace("History + ", "", regex=False)

axes[0].barh(plot_df["Label"], plot_df["RMSE"])
axes[0].invert_yaxis()
axes[0].set_title("RMSE by model and feature set (lower is better)")
axes[0].set_xlabel("RMSE (mm)")

axes[1].barh(plot_df["Label"], plot_df["MAE"])
axes[1].invert_yaxis()
axes[1].set_title("MAE by model and feature set (lower is better)")
axes[1].set_xlabel("MAE (mm)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m5_01_regression_model_comparison.png"), dpi=160, bbox_inches="tight")
plt.show()

## Step 9 — Actual vs predicted rainfall and residual analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, best_pred, alpha=0.35, s=18)
max_val = max(float(y_test.max()), float(best_pred.max()))
axes[0].plot([0, max_val], [0, max_val], linestyle="--")
axes[0].set(xlabel="Actual next-day rainfall (mm)", ylabel="Predicted rainfall (mm)", title="Actual vs predicted rainfall")

residuals = y_test.to_numpy() - best_pred
sns.histplot(residuals, bins=40, kde=True, ax=axes[1])
axes[1].axvline(0, linestyle="--")
axes[1].set(xlabel="Residual (actual − predicted), mm", title="Residual distribution")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "m5_02_actual_predicted_residuals.png"), dpi=160, bbox_inches="tight")
plt.show()

## Step 10 — Inspect performance by rainfall intensity
Rainfall is usually zero or small on many days, so a single overall score can hide poor performance on heavy-rain days. This table reports errors by the **actual next-day rainfall** range. It is diagnostic only; it does not change the trained model.

In [ ]:
diagnostics = test_df[["state", "district", "date", "target_date", TARGET]].copy()
diagnostics["predicted_mm"] = best_pred
diagnostics["absolute_error_mm"] = np.abs(diagnostics[TARGET] - diagnostics["predicted_mm"])

bins = [-0.001, 0, 2.5, 15.5, 64.4, np.inf]
labels = ["No rain", "Very light", "Light", "Moderate", "Heavy / above"]
diagnostics["actual_rain_band"] = pd.cut(diagnostics[TARGET], bins=bins, labels=labels)
by_band = diagnostics.groupby("actual_rain_band", observed=False).agg(
    observations=(TARGET, "size"),
    actual_mean_mm=(TARGET, "mean"),
    predicted_mean_mm=("predicted_mm", "mean"),
    MAE_mm=("absolute_error_mm", "mean")
)
display(by_band.round(3))

## Step 11 — Save the best model, predictions and metrics
The saved model is a complete sklearn pipeline, including its imputer and (for Linear Regression) scaler. The metadata records the feature order and split date so later modules can load it consistently.

In [ ]:
MODEL_PATH = os.path.join(MODEL_DIR, "rainfall_regression_model.pkl")
METRICS_PATH = os.path.join(MODEL_DIR, "regression_metrics.csv")
PRED_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "regression_test_predictions.csv")
META_PATH = os.path.join(MODEL_DIR, "regression_model_metadata.json")

joblib.dump(best_model, MODEL_PATH)
results_df.to_csv(METRICS_PATH, index=False)
diagnostics.to_csv(PRED_PATH, index=False)

metadata = {
    "project": "RainGuard",
    "task": "next-calendar-day district rainfall regression",
    "target": TARGET,
    "best_model": best_key[1],
    "feature_set": best_key[0],
    "features": best_features,
    "split_date": str(split_date.date()),
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "metrics": {k: float(best_row[k]) for k in ["MAE", "MSE", "RMSE", "R2"]},
    "prediction_clip": "negative predictions clipped to zero mm",
}
with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved model:", MODEL_PATH)
print("Saved metrics:", METRICS_PATH)
print("Saved test predictions:", PRED_PATH)
print("Saved metadata:", META_PATH)
print("Saved figures:", os.path.join(FIG_DIR, "m5_01_regression_model_comparison.png"), "and", os.path.join(FIG_DIR, "m5_02_actual_predicted_residuals.png"))

## Module 5 summary and interpretation

Use the actual output from this notebook to complete this summary after running all cells:

- Best algorithm: **[fill from Step 7]**
- Best feature set: **[fill from Step 7]**
- MAE: **[fill from Step 7] mm**
- RMSE: **[fill from Step 7] mm**
- R²: **[fill from Step 7]**

**Interpretation:** MAE is the average absolute prediction error in millimetres. RMSE penalizes larger errors more strongly. R² describes the proportion of test-target variation explained by the model; it can be negative if the model performs worse than predicting the test mean.

**Limitations:** The dataset covers only 73 days, so this is a short-horizon demonstration, not a model validated across India's full annual monsoon cycle. Rainfall is spatially variable and often zero; errors on heavy-rain observations must be reported separately. A rainfall forecast is not a flood forecast. Flood prediction requires flood occurrence/extent labels and additional variables such as river level, drainage, soil saturation and terrain.

**Next:** Module 6 — Rainfall intensity classification.